# Compare previous protocols
- To quantify baseline fitting issue.
- 1 minute baseline window vs 30 minute baseline window
- 1 minute baseline window is the current dff.

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from scipy.stats import skew
import jax
import jax.numpy as jnp
from tqdm.auto import tqdm
from joblib import Parallel, delayed

from lamf_analysis.code_ocean import capsule_data_utils as cdu
from lamf_analysis.code_ocean import docdb_utils
from aind_ophys_utils import dff as dff_utils
from aind_ophys_utils.signal_utils import noise_std

from baseline_fitting import AsymmetricTukeyBiweight, fit_baseline 

%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.close_figures = True



In [4]:
data_dir = Path('/root/capsule/data')
subject_ids = [755252, 767018, 767022, 783551, 785054, 782149, 788406, 790322, 800792, 800995, 804363, 804670] # all Slc32a1;Oi1 collected so far


In [5]:
def get_attach_df(subject_id):
    # including single-cell-zdrift
    session_infos = docdb_utils.get_session_infos_from_docdb(subject_id, filter_test_data=True)
    processed_infos = docdb_utils.get_processed_data_info(subject_id).sort_values('long_window').drop_duplicates(subset=['raw_name'])
    merged_df = processed_infos.merge(session_infos, left_on='raw_name', right_on='raw_asset_name', how='left')

    sczdrift_df = docdb_utils.get_derived_data_assets(subject_id, 'single-cell-zdrift-qc')
    sczdrift_df.rename(columns={'derived_name': 'single_cell_zdrift_derived_name',
                                'derived_asset_id': 'single_cell_zdrift_derived_asset_id'}, inplace=True)
    merged_df = merged_df.merge(sczdrift_df[['raw_name',
                                            'single_cell_zdrift_derived_name',
                                            'single_cell_zdrift_derived_asset_id']],
                                on='raw_name', how='inner')
    # attach_df = merged_df[merged_df.session_type.str.contains('OPHYS_')]
    return merged_df

In [6]:
subject_id = subject_ids[0]
merged_df = get_attach_df(subject_id).sort_values('acquisition_date').reset_index(drop=True)
merged_df.head()

,raw_name,long_window,processed_asset_id,processed_date,processed_name,acquisition_date,session_type,rig_id,session_key,project_name,raw_asset_name,raw_asset_id,s3_path,session_type_exposures,single_cell_zdrift_derived_name,single_cell_zdrift_derived_asset_id
0,multiplane-ophys_755252_2024-11-12_09-43-51,60,6747f968-d6dc-4db6-b6a6-941aac40425e,2025-09-05,multiplane-ophys_755252_2024-11-12_09-43-51_pr...,2024-11-12,TRAINING_0_gratings_autorewards_15min,MESO.1,755252_2024-11-12,Learning mFISH-V1omFISH,multiplane-ophys_755252_2024-11-12_09-43-51,73d47ce2-1d56-48db-9f7b-d99aaf782d1d,s3://aind-private-data-prod-o5171v/multiplane-...,1,multiplane-ophys_755252_2024-11-12_09-43-51_si...,b61cd61a-2aa1-47fb-8619-dc64e15326e7
1,multiplane-ophys_755252_2024-11-13_09-16-29,60,41d3bfb3-f969-4442-936d-58aaf02720d6,2025-09-05,multiplane-ophys_755252_2024-11-13_09-16-29_pr...,2024-11-13,TRAINING_1_gratings,MESO.1,755252_2024-11-13,Learning mFISH-V1omFISH,multiplane-ophys_755252_2024-11-13_09-16-29,207fb83f-3fdf-4879-8f45-f02119557d4e,s3://aind-private-data-prod-o5171v/multiplane-...,1,multiplane-ophys_755252_2024-11-13_09-16-29_si...,4af41f88-0f0a-4909-91d4-4b099724ff11
2,multiplane-ophys_755252_2024-11-14_11-28-00,60,15f5d46e-80d5-422f-8608-461484b4c86b,2025-09-05,multiplane-ophys_755252_2024-11-14_11-28-00_pr...,2024-11-14,TRAINING_1_gratings,MESO.1,755252_2024-11-14,Learning mFISH-V1omFISH,multiplane-ophys_755252_2024-11-14_11-28-00,1b187fe7-13ac-4800-9dba-b6523f22765c,s3://aind-private-data-prod-o5171v/multiplane-...,2,multiplane-ophys_755252_2024-11-14_11-28-00_si...,8c939675-da89-4505-8288-2045c79eff8a
3,multiplane-ophys_755252_2024-11-15_10-49-40,60,f0c72bdf-844e-4d27-bb71-c6d9e879788c,2025-09-05,multiplane-ophys_755252_2024-11-15_10-49-40_pr...,2024-11-15,TRAINING_1_gratings,MESO.1,755252_2024-11-15,Learning mFISH-V1omFISH,multiplane-ophys_755252_2024-11-15_10-49-40,95ba6dcc-c32a-4812-a234-27ae729cc497,s3://aind-private-data-prod-o5171v/multiplane-...,3,multiplane-ophys_755252_2024-11-15_10-49-40_si...,97ea2b7a-deed-492d-9c3d-9eb9d858a15d
4,multiplane-ophys_755252_2024-11-18_08-01-08,60,bccb963f-a624-4ce9-8f67-7faafeb0846e,2025-09-05,multiplane-ophys_755252_2024-11-18_08-01-08_pr...,2024-11-18,TRAINING_1_gratings,MESO.1,755252_2024-11-18,Learning mFISH-V1omFISH,multiplane-ophys_755252_2024-11-18_08-01-08,5f5ffe30-e764-41db-883b-c00c04219413,s3://aind-private-data-prod-o5171v/multiplane-...,4,multiplane-ophys_755252_2024-11-18_08-01-08_si...,4c304ef7-5ed3-4f50-8cb4-b021ed89fb74


In [5]:
session_i = 10
session_row = merged_df.iloc[session_i]
processed_name = session_row.processed_name
processed_dir = data_dir / processed_name
plane_ids = cdu.get_plane_ids_from_processed_path(processed_dir)
intended_depths = [cdu.get_intended_depth(processed_dir / plane_id) for plane_id in plane_ids]

sczdrift_name = session_row.single_cell_zdrift_derived_name
sczdrift_path = data_dir / sczdrift_name

No intended depth found in platform info for /root/capsule/data/multiplane-ophys_755252_2024-12-05_11-34-40_processed_2025-09-05_11-46-59/VISp_0, returning targeted depth as fallback
No intended depth found in platform info for /root/capsule/data/multiplane-ophys_755252_2024-12-05_11-34-40_processed_2025-09-05_11-46-59/VISp_1, returning targeted depth as fallback
No intended depth found in platform info for /root/capsule/data/multiplane-ophys_755252_2024-12-05_11-34-40_processed_2025-09-05_11-46-59/VISp_2, returning targeted depth as fallback
No intended depth found in platform info for /root/capsule/data/multiplane-ophys_755252_2024-12-05_11-34-40_processed_2025-09-05_11-46-59/VISp_3, returning targeted depth as fallback
No intended depth found in platform info for /root/capsule/data/multiplane-ophys_755252_2024-12-05_11-34-40_processed_2025-09-05_11-46-59/VISp_4, returning targeted depth as fallback
No intended depth found in platform info for /root/capsule/data/multiplane-ophys_7552

In [6]:
F_all = []
dff_short_window_all = []
dff_long_window_all = []
baseline_short_window_all = []
baseline_long_window_all = []
valid_roi_inds_all = []
sczdrift_df_all_list = []
for plane_id in plane_ids:
    plane_path = processed_dir / plane_id
    plane_depth = cdu.get_intended_depth(plane_path)
    roi_table = cdu.get_roi_table_from_plane_path(plane_path)
    valid_roi_inds = roi_table.query('valid_roi').cell_roi_id.values

    sczdrift_df_fn = next((sczdrift_path / plane_id).glob('*roi_time_profile.csv'))
    sczdrift_df = pd.read_csv(sczdrift_df_fn)
    zdrift_um = (sczdrift_df.matched_plane_index_smoothed.max() - sczdrift_df.matched_plane_index_smoothed.min()) * 0.75
    max_frame_num = sczdrift_df.smoothed_frame_index.max()
    sczdrift_df = sczdrift_df.query('smoothed_frame_index == @max_frame_num')[['cell_roi_id', 'fractional_change_from_first_frame']]
    sczdrift_df['plane_id'] = plane_id
    sczdrift_df['intended_depth'] = plane_depth
    sczdrift_df['z_drift_um'] = zdrift_um

    assert np.all(valid_roi_inds == sczdrift_df.cell_roi_id.unique()), f"Mismatch between valid_roi_inds and sczdrift_df cell_roi_id for plane {plane_id}"

    valid_roi_inds_all.append(valid_roi_inds)
    sczdrift_df_all_list.append(sczdrift_df)    
    
    F = cdu.load_corrected_fluorescence(plane_path=plane_path)
    F_valid = F[valid_roi_inds, :]
    frame_rate = cdu.get_frame_rate_from_plane_path(plane_path)
    dff_short = cdu.load_dff_from_plane_path(plane_path)
    baseline_short = cdu.get_baseline_traces(plane_path)
    dff_long, baseline_long, _ = dff_utils.dff(F_valid, long_window=60*30, fs=frame_rate)
        
    F_all.append(F_valid)
    dff_short_window_all.append(dff_short[valid_roi_inds, :])
    dff_long_window_all.append(dff_long)
    baseline_short_window_all.append(baseline_short[valid_roi_inds, :])
    baseline_long_window_all.append(baseline_long)

F_all_array = np.concatenate(F_all, axis=0)
dff_short_window_all_array = np.concatenate(dff_short_window_all, axis=0)
dff_long_window_all_array = np.concatenate(dff_long_window_all, axis=0)
baseline_short_window_all_array = np.concatenate(baseline_short_window_all, axis=0)
baseline_long_window_all_array = np.concatenate(baseline_long_window_all, axis=0)
sczdrift_df_all = pd.concat(sczdrift_df_all_list, ignore_index=True)

F_noise = noise_std(F_all_array, 'mad')
F_signal = np.percentile(F_all_array - baseline_short_window_all_array, 99, axis=1)
F_snr = F_signal / F_noise
F_skewness = skew(F_all_array, axis=1)

No intended depth found in platform info for /root/capsule/data/multiplane-ophys_755252_2024-12-05_11-34-40_processed_2025-09-05_11-46-59/VISp_0, returning targeted depth as fallback
Using provided plane_path to load corrected fluorescence
No intended depth found in platform info for /root/capsule/data/multiplane-ophys_755252_2024-12-05_11-34-40_processed_2025-09-05_11-46-59/VISp_1, returning targeted depth as fallback
Using provided plane_path to load corrected fluorescence
No intended depth found in platform info for /root/capsule/data/multiplane-ophys_755252_2024-12-05_11-34-40_processed_2025-09-05_11-46-59/VISp_2, returning targeted depth as fallback
Using provided plane_path to load corrected fluorescence
No intended depth found in platform info for /root/capsule/data/multiplane-ophys_755252_2024-12-05_11-34-40_processed_2025-09-05_11-46-59/VISp_3, returning targeted depth as fallback
Using provided plane_path to load corrected fluorescence
No intended depth found in platform info

In [8]:
bleaching_window = int(frame_rate * 60 * 5)  # 5 minute window

baseline_diff = baseline_short_window_all_array - baseline_long_window_all_array
zscored_baseline_diff = baseline_diff / np.std(baseline_diff, axis=1, keepdims=True)

bleaching_metric = np.mean(zscored_baseline_diff[:, :bleaching_window], axis=1)
sustained_metric = np.mean(zscored_baseline_diff[:, bleaching_window*2:], axis=1)

b_inits = np.mean(F_all_array - baseline_long_window_all_array, axis=1)

In [ ]:
i = 470
fig, ax  = plt.subplots()
ax.plot(F_all_array[i, :], label='F')
ax.plot(baseline_short_window_all_array[i, :], label='baseline_short')
ax.plot(baseline_long_window_all_array[i, :], label='baseline_long')
ax.legend()
ax.set_title(f'Cell {i}, Skewness: {F_skewness[i]:.2f}, SNR: {F_snr[i]:.2f}')
# ax.set_xlim(0, 500)

In [ ]:
fig, ax = plt.subplots()

ax.imshow(zscored_baseline_diff, aspect='auto')

In [ ]:
i = 512
fig, ax  = plt.subplots()
ax.plot(F_all_array[i, :], label='F')
ax.plot(baseline_short_window_all_array[i, :], label='baseline_short')
ax.plot(baseline_long_window_all_array[i, :], label='baseline_long')
ax.legend()
ax.set_title(f'Cell {i}, Skewness: {F_skewness[i]:.2f}, SNR: {F_snr[i]:.2f}')
ax.set_xlim(0, 3000)

In [6]:
# run nonlinear_fitting - basic

def model(params, t, xp=np):
    """
    Baseline with biphasic decay and saturating brightening.

    Parameters
    ----------
    params : np.ndarray
        Parameter vector: [b_inf, b_slow, b_fast, b_bright,
                           t_slow, t_fast, t_bright]
    t : np.ndarray
        Timestamps

    Returns
    -------
    y : np.ndarray | jax.Array
        Model prediction
    """
    b_inf, b_slow, b_fast, b_bright, t_slow, t_fast, t_bright = params
    E_slow = xp.exp(-t / t_slow)
    E_fast = xp.exp(-t / t_fast)
    E_bright = xp.exp(-t / t_bright)
    return b_inf + b_slow * E_slow + b_fast * E_fast - b_bright * E_bright

In [ ]:
# M=AsymmetricTukeyBiweight(2, 3)
# from tqdm.auto import tqdm
# timestamps = np.arange(F_all_array.shape[1]) / frame_rate
# t_max = timestamps[-1]
# t_high_bound = t_max* 5
# F0trend_all = []
# F0_all = []
# res_all = []
# loss_all = []
# for i, F in tqdm(enumerate(F_all_array), total=F_all_array.shape[0]):
#     F0, F0trend, res, info = fit_baseline(
#         F, timestamps, model,
#         # initial parameters
#         [F.mean(),      # b_inf
#          b_inits[i],    # b_slow
#          b_inits[i],    # b_fast
#          b_inits[i],    # b_bright
#          t_max/2,       # t_slow
#          60,            # t_fast
#          t_max/2],      # t_bright
#         bounds=[
#             (0, None),    # b_inf
#             (0, None),    # b_slow
#             (0, None),    # b_fast
#             (0, None),    # b_bright
#             (300, t_high_bound),  # t_slow
#             (1, 300),     # t_fast  <-- constrained to < 5 min
#             (300, t_high_bound),    # t_bright
#             ],
#         M=M,
#         fixed_sigma=F_noise[i],
#         backend="jax", dtype=jnp.float32)
#     ls = info['lowess_sigma']*F0trend
#     loss = np.mean(M.rho((F - F0) / ls) * ls**2)
#     F0trend_all.append(F0trend)
#     F0_all.append(F0)
#     res_all.append(res)
#     loss_all.append(loss)

In [7]:
def get_initial_f_results(row):
    processed_name = row.processed_name
    processed_dir = data_dir / processed_name
    plane_ids = cdu.get_plane_ids_from_processed_path(processed_dir)

    sczdrift_name = row.single_cell_zdrift_derived_name
    sczdrift_path = data_dir / sczdrift_name
    F_all = []
    dff_short_window_all = []
    dff_long_window_all = []
    baseline_short_window_all = []
    baseline_long_window_all = []
    valid_roi_inds_all = []
    sczdrift_df_all_list = []
    for plane_id in plane_ids:
        plane_path = processed_dir / plane_id
        plane_depth = cdu.get_intended_depth(plane_path)
        roi_table = cdu.get_roi_table_from_plane_path(plane_path)
        valid_roi_inds = roi_table.query('valid_roi').cell_roi_id.values

        sczdrift_df_fn = next((sczdrift_path / plane_id).glob('*roi_time_profile.csv'))
        sczdrift_df = pd.read_csv(sczdrift_df_fn)
        zdrift_um = (sczdrift_df.matched_plane_index_smoothed.max() - sczdrift_df.matched_plane_index_smoothed.min()) * 0.75
        max_frame_num = sczdrift_df.smoothed_frame_index.max()
        sczdrift_df = sczdrift_df.query('smoothed_frame_index == @max_frame_num')[['cell_roi_id', 'fractional_change_from_first_frame']]
        sczdrift_df['plane_id'] = plane_id
        sczdrift_df['intended_depth'] = plane_depth
        sczdrift_df['intended_depth'] = plane_depth
        sczdrift_df['z_drift_um'] = zdrift_um

        assert np.all(valid_roi_inds == sczdrift_df.cell_roi_id.unique()), f"Mismatch between valid_roi_inds and sczdrift_df cell_roi_id for plane {plane_id}"

        valid_roi_inds_all.append(valid_roi_inds)
        sczdrift_df_all_list.append(sczdrift_df)    
        
        F = cdu.load_corrected_fluorescence(plane_path=plane_path)
        F_valid = F[valid_roi_inds, :]
        frame_rate = cdu.get_frame_rate_from_plane_path(plane_path)
        dff_short = cdu.load_dff_from_plane_path(plane_path)
        baseline_short = cdu.get_baseline_traces(plane_path)
        dff_long, baseline_long, _ = dff_utils.dff(F_valid, long_window=60*30, fs=frame_rate)
            
        F_all.append(F_valid)
        dff_short_window_all.append(dff_short[valid_roi_inds, :])
        dff_long_window_all.append(dff_long)
        baseline_short_window_all.append(baseline_short[valid_roi_inds, :])
        baseline_long_window_all.append(baseline_long)

    F_all_array = np.concatenate(F_all, axis=0)
    dff_short_window_all_array = np.concatenate(dff_short_window_all, axis=0)
    dff_long_window_all_array = np.concatenate(dff_long_window_all, axis=0)
    baseline_short_window_all_array = np.concatenate(baseline_short_window_all, axis=0)
    baseline_long_window_all_array = np.concatenate(baseline_long_window_all, axis=0)
    sczdrift_df_all = pd.concat(sczdrift_df_all_list, ignore_index=True)

    F_noise = noise_std(F_all_array, 'mad')
    F_signal = np.percentile(F_all_array - baseline_short_window_all_array, 99, axis=1)
    F_snr = F_signal / F_noise
    F_skewness = skew(F_all_array, axis=1)

    return F_all_array, dff_short_window_all_array, dff_long_window_all_array, \
        baseline_short_window_all_array, baseline_long_window_all_array, \
        sczdrift_df_all, F_noise, F_signal, F_snr, F_skewness, frame_rate

In [8]:
save_dir_base = Path('/root/capsule/scratch/first_try')

M = AsymmetricTukeyBiweight(2, 3)

nonlinear_fitting_settings = {
    'model': model,
    'M-estimator': M,
}

In [ ]:

files_list = ['F_all_array.npy', 'dff_short_window_all_array.npy', 'dff_long_window_all_array.npy',
                'baseline_short_window_all_array.npy', 'baseline_long_window_all_array.npy',
                'sczdrift_df_all.csv', 'F_noise.npy', 'F_signal.npy', 'F_snr.npy', 'F_skewness.npy',
                'timestamps.npy', 'bleaching_metric.npy', 'sustained_metric.npy']

for i, row in tqdm(merged_df.iterrows(), total=merged_df.shape[0]):
    session_key = row.session_key
    save_dir = save_dir_base / session_key

    if all((save_dir / fn).exists() for fn in files_list):
        print(f"All files for session {session_key} already exist, skipping...")
        continue

    # saving images and roi tables
    # These are not checked by the files_list (cheap to rerun if necessary)
    processed_path = data_dir / row.processed_name
    plane_ids = cdu.get_plane_ids_from_processed_path(processed_path)
    for plane_id in plane_ids:
        plane_path = processed_path / plane_id
        roi_table = cdu.get_roi_table_from_plane_path(plane_path)
        mean_image = cdu.load_projection_image(plane_path, 'mean')
        max_image = cdu.load_projection_image(plane_path, 'max')
        np.save(save_dir / f"{plane_id}_mean_img.npy", mean_image)
        np.save(save_dir / f"{plane_id}_max_img.npy", max_image)
        roi_table.to_pickle(save_dir / f"{plane_id}_roi_table.pkl")

    # Main loading and calculation
    F_all_array, dff_short_window_all_array, dff_long_window_all_array, \
        baseline_short_window_all_array, baseline_long_window_all_array, \
        sczdrift_df_all, F_noise, F_signal, F_snr, F_skewness, frame_rate = \
            get_initial_f_results(row)
    
    bleaching_window = int(frame_rate * 60 * 5)  # 5 minute window

    baseline_diff = baseline_short_window_all_array - baseline_long_window_all_array
    zscored_baseline_diff = baseline_diff / np.std(baseline_diff, axis=1, keepdims=True)

    bleaching_metric = np.mean(zscored_baseline_diff[:, :bleaching_window], axis=1)
    sustained_metric = np.mean(zscored_baseline_diff[:, bleaching_window*2:], axis=1)

    b_inits = np.mean(F_all_array - baseline_long_window_all_array, axis=1)

    timestamps = np.arange(F_all_array.shape[1]) / frame_rate
    t_max = timestamps[-1]
    t_high_bound = t_max * 5

    def _fit_one(F, b_init, noise):
        F0, F0trend, res, info = fit_baseline(
            F, timestamps, model,
            [F.mean(), b_init, b_init, b_init,
            t_max/2, 60, t_max/2],
            bounds=[
                (0, None),
                (0, None),
                (0, None),
                (0, None),
                (300, t_high_bound),
                (1, 300),
                (300, t_high_bound),
            ],
            M=M,
            fixed_sigma=noise,
            backend="jax", dtype=jnp.float32)
        ls = info['lowess_sigma'] * F0trend
        loss = np.mean(M.rho((F - F0) / ls) * ls**2)
        return F0trend, F0, res, loss

    results = Parallel(n_jobs=-1, backend='loky')(
        delayed(_fit_one)(F, b_inits[i], F_noise[i])
        for i, F in enumerate(F_all_array)
    )

    F0trend_all, F0_all, res_all, loss_all = map(list, zip(*results))
    # save all the data
    save_dir.mkdir(parents=True, exist_ok=True)
    np.save(save_dir / "F0trend_all.npy", F0trend_all)
    np.save(save_dir / "F0_all.npy", F0_all)
    np.save(save_dir / "res_all.npy", res_all)
    np.save(save_dir / "loss_all.npy", loss_all)
    np.save(save_dir / "F_all_array.npy", F_all_array)
    np.save(save_dir / "dff_short_window_all_array.npy", dff_short_window_all_array)
    np.save(save_dir / "dff_long_window_all_array.npy", dff_long_window_all_array)
    np.save(save_dir / "baseline_short_window_all_array.npy", baseline_short_window_all_array)
    np.save(save_dir / "baseline_long_window_all_array.npy", baseline_long_window_all_array)
    sczdrift_df_all.to_csv(save_dir / "sczdrift_df_all.csv", index=False)
    np.save(save_dir / "F_noise.npy", F_noise)
    np.save(save_dir / "F_signal.npy", F_signal)
    np.save(save_dir / "F_snr.npy", F_snr)
    np.save(save_dir / "F_skewness.npy", F_skewness)
    np.save(save_dir / "timestamps.npy", timestamps)
    np.save(save_dir / "bleaching_metric.npy", bleaching_metric)
    np.save(save_dir / "sustained_metric.npy", sustained_metric)


  0%|          | 0/26 [00:00<?, ?it/s]

All files for session 755252_2024-11-12 already exist, skipping...
All files for session 755252_2024-11-13 already exist, skipping...
All files for session 755252_2024-11-14 already exist, skipping...
All files for session 755252_2024-11-15 already exist, skipping...
All files for session 755252_2024-11-18 already exist, skipping...
All files for session 755252_2024-11-19 already exist, skipping...
All files for session 755252_2024-11-21 already exist, skipping...
All files for session 755252_2024-11-22 already exist, skipping...
All files for session 755252_2024-12-03 already exist, skipping...
All files for session 755252_2024-12-04 already exist, skipping...
All files for session 755252_2024-12-05 already exist, skipping...
All files for session 755252_2024-12-06 already exist, skipping...
All files for session 755252_2024-12-09 already exist, skipping...
All files for session 755252_2024-12-10 already exist, skipping...
All files for session 755252_2024-12-11 already exist, skippin

/opt/conda/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


No intended depth found in platform info for /root/capsule/data/multiplane-ophys_755252_2025-01-13_09-33-41_processed_2025-09-05_18-25-53/VISp_0, returning targeted depth as fallback
Using provided plane_path to load corrected fluorescence
No intended depth found in platform info for /root/capsule/data/multiplane-ophys_755252_2025-01-13_09-33-41_processed_2025-09-05_18-25-53/VISp_1, returning targeted depth as fallback
Using provided plane_path to load corrected fluorescence
No intended depth found in platform info for /root/capsule/data/multiplane-ophys_755252_2025-01-13_09-33-41_processed_2025-09-05_18-25-53/VISp_2, returning targeted depth as fallback
Using provided plane_path to load corrected fluorescence
No intended depth found in platform info for /root/capsule/data/multiplane-ophys_755252_2025-01-13_09-33-41_processed_2025-09-05_18-25-53/VISp_3, returning targeted depth as fallback
Using provided plane_path to load corrected fluorescence
No intended depth found in platform info

/opt/conda/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


No intended depth found in platform info for /root/capsule/data/multiplane-ophys_755252_2025-01-14_12-09-40_processed_2025-09-05_20-42-15/VISp_0, returning targeted depth as fallback
Using provided plane_path to load corrected fluorescence
No intended depth found in platform info for /root/capsule/data/multiplane-ophys_755252_2025-01-14_12-09-40_processed_2025-09-05_20-42-15/VISp_1, returning targeted depth as fallback
Using provided plane_path to load corrected fluorescence
No intended depth found in platform info for /root/capsule/data/multiplane-ophys_755252_2025-01-14_12-09-40_processed_2025-09-05_20-42-15/VISp_2, returning targeted depth as fallback
Using provided plane_path to load corrected fluorescence
No intended depth found in platform info for /root/capsule/data/multiplane-ophys_755252_2025-01-14_12-09-40_processed_2025-09-05_20-42-15/VISp_3, returning targeted depth as fallback
Using provided plane_path to load corrected fluorescence
No intended depth found in platform info

In [ ]:
for i, row in tqdm(merged_df.iterrows(), total=merged_df.shape[0]):
    session_key = row.session_key
    save_dir = save_dir_base / session_key

    processed_path = data_dir / row.processed_name
    plane_ids = cdu.get_plane_ids_from_processed_path(processed_path)
    for plane_id in plane_ids:
        plane_path = processed_path / plane_id
        roi_table = cdu.get_roi_table_from_plane_path(plane_path)
        mean_image = cdu.load_projection_image(plane_path, 'mean')
        max_image = cdu.load_projection_image(plane_path, 'max')
        np.save(save_dir / f"{plane_id}_mean_img.npy", mean_image)
        np.save(save_dir / f"{plane_id}_max_img.npy", max_image)
        roi_table.to_pickle(save_dir / f"{plane_id}_roi_table.pkl")


  0%|          | 0/26 [00:00<?, ?it/s]